In [1]:
import os
import json
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
import torch

In [2]:
IMG_DIR = "image"
LABEL_FILE = "all_image_label.xlsx"
TRAIN_FILE = "train.json"
DEV_FILE = "dev.json"
TEST_FILE = "test.json"

In [3]:
label_df = pd.read_excel(LABEL_FILE)
label_columns = ["Location", "Food", "Room", "Facilities", "Service", "Public_area"]
print(f"Các cột trong file Excel: {list(label_df.columns)}")

Các cột trong file Excel: ['file_name', 'Location', 'Food', 'Room', 'Facilities', 'Service', 'Public_area']


In [4]:
def build_label(row):
    return [int(row[col]) if not pd.isna(row[col]) else 0 for col in label_columns]

label_map = {
    row["file_name"]: build_label(row)
    for _, row in label_df.iterrows()
}

In [5]:
def load_image_paths_and_labels(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    samples = []
    for item in data:
        for filename in item.get("list_img", []):
            label = label_map.get(filename)
            if label is not None:
                samples.append((os.path.join(IMG_DIR, filename), label))
    return samples

In [6]:
class ImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label, dtype=torch.float32)
        return image, label

In [7]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [8]:
train_data = load_image_paths_and_labels(TRAIN_FILE)
dev_data = load_image_paths_and_labels(DEV_FILE)
test_data = load_image_paths_and_labels(TEST_FILE)

train_dataset = ImageDataset(train_data, transform=transform)
dev_dataset = ImageDataset(dev_data, transform=transform)
test_dataset = ImageDataset(test_data, transform=transform)

In [9]:
print(f"Tổng số ảnh train: {len(train_dataset)}")
print(f"Tổng số ảnh dev: {len(dev_dataset)}")
print(f"Tổng số ảnh test: {len(test_dataset)}")
image, label = train_dataset[0]
print("Kích thước ảnh:", image.shape)
print("Nhãn:", label)

Tổng số ảnh train: 5428
Tổng số ảnh dev: 1789
Tổng số ảnh test: 1841
Kích thước ảnh: torch.Size([3, 128, 128])
Nhãn: tensor([0., 0., 0., 1., 0., 1.])
